In [2]:
# Cell 1 – Load & clean
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv('../data/OnlineRetail.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancellations, nulls, negatives
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df.dropna(subset=['CustomerID'], inplace=True)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['CustomerID'] = df['CustomerID'].astype(int)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(df.shape)

(397884, 9)


In [3]:
# Cell 2 – RFM feature engineering
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

rfm.to_csv('../data/rfm.csv', index=False)
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [4]:
# Cell 3 – Rolling statistics (for time-series later)
daily = df.groupby(df['InvoiceDate'].dt.date)['Revenue'].sum().reset_index()
daily.columns = ['Date','Revenue']
daily['Rolling7']  = daily['Revenue'].rolling(7).mean()
daily['Rolling30'] = daily['Revenue'].rolling(30).mean()
daily.to_csv('../data/daily_sales.csv', index=False)
daily.tail()

,Date,Revenue,Rolling7,Rolling30
300,2011-12-05,58202.21,44284.347143,44321.441000
301,2011-12-06,46144.04,43471.828571,44888.482000
302,2011-12-07,69354.21,46400.761429,45909.132333
303,2011-12-08,50519.41,47691.930000,45570.709000
304,2011-12-09,184349.28,67665.542857,49845.710333
